In [2]:
import pandas as pd
import numpy as np

# Creating a representative phishing dataset from scratch
# Real phishing emails follow these exact patterns

phishing_emails = [
    "Urgent: Your account has been suspended. Verify your credentials immediately at bit.ly/secure-login",
    "WINNER! You have been selected for a $1000 gift card. Click here to claim now before it expires",
    "Your PayPal account needs immediate attention. Confirm your password to avoid suspension",
    "Dear customer, your bank account has been compromised. Enter your details to secure it now",
    "Final notice: Your Netflix subscription will be cancelled. Update payment information immediately",
    "Congratulations! You won an iPhone 15. Click here to verify your shipping address now",
    "Security Alert: Unauthorized login detected. Verify your identity immediately or account will be locked",
    "Your Amazon order has been suspended. Confirm your credentials at amazon-secure-login.net",
    "URGENT: IRS tax refund pending. Submit your social security number to claim $3,240 refund",
    "Your email storage is 99% full. Click here to upgrade immediately or lose your emails",
    "Dear user, verify your account within 24 hours or it will be permanently deleted",
    "You have a pending wire transfer of $5000. Confirm your bank details to receive funds",
    "Microsoft security team: Your computer is infected. Call 1-800-xxx-xxxx immediately",
    "Your Apple ID has been locked. Verify your information at apple-id-secure.com now",
    "Exclusive offer expires tonight: Enter credit card details to claim your free prize",
    "Action required: Unusual sign-in activity detected. Confirm your password immediately",
    "Your DHL package is on hold. Pay $2.99 customs fee at this link to release delivery",
    "Verify your account now or lose access permanently. Click here within 24 hours",
    "You have been selected for a government grant. Submit personal details to claim $10,000",
    "Dear valued customer, your account will be suspended unless you update billing information",
]

safe_emails = [
    "Hey, can we move the meeting to 3pm tomorrow? Let me know if that works for you",
    "Please find attached the quarterly report for your review. Let me know if you have questions",
    "Thanks for joining us at the conference. It was great meeting you. Let's stay in touch",
    "Reminder: Team standup at 10am today. Agenda attached for reference",
    "Your order #12345 has shipped and will arrive by Thursday. Track it on our website",
    "Hi, I wanted to follow up on the project proposal we discussed last week",
    "The new version of the software is now available. Release notes are in the documentation",
    "Happy birthday! Hope you have a wonderful day and a great year ahead",
    "Please review the attached contract and send back your comments by end of week",
    "Your appointment is confirmed for next Monday at 2pm. See you then",
    "Great work on the presentation today. The client was really impressed with your analysis",
    "I have attached the invoice for last month services. Payment due within 30 days",
    "Can you share the latest version of the spreadsheet when you get a chance",
    "The team lunch is scheduled for Friday at noon. Restaurant details in the calendar invite",
    "Just checking in to see how the new project is going. Need any help from my end",
    "Your subscription has been renewed successfully. Receipt attached for your records",
    "Thanks for your feedback on the product. We will look into the issue you mentioned",
    "The conference call has been rescheduled to 4pm. Dial-in details remain the same",
    "Please complete the annual compliance training by end of month. Link in HR portal",
    "Good news — the budget has been approved for Q2. We can proceed with the project",
]

# Create DataFrame
texts = phishing_emails + safe_emails
labels = [1] * len(phishing_emails) + [0] * len(safe_emails)  # 1=phishing, 0=safe

df = pd.DataFrame({"text": texts, "label": labels})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle
print(df)
print(f"Total emails: {len(df)}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())
print(f"\nSample phishing email:")
print(df[df['label']==1].iloc[0]['text'])
print(f"\nSample safe email:")
print(df[df['label']==0].iloc[0]['text'])

                                                 text  label
0   Dear valued customer, your account will be sus...      1
1   Your DHL package is on hold. Pay $2.99 customs...      1
2   Action required: Unusual sign-in activity dete...      1
3   The new version of the software is now availab...      0
4   Final notice: Your Netflix subscription will b...      1
5   Microsoft security team: Your computer is infe...      1
6   The conference call has been rescheduled to 4p...      0
7   Happy birthday! Hope you have a wonderful day ...      0
8   Good news — the budget has been approved for Q...      0
9   Security Alert: Unauthorized login detected. V...      1
10  Hi, I wanted to follow up on the project propo...      0
11  Your email storage is 99% full. Click here to ...      1
12  Your Apple ID has been locked. Verify your inf...      1
13  I have attached the invoice for last month ser...      0
14  Just checking in to see how the new project is...      0
15  URGENT: IRS tax refu

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


In [4]:
# ---- STEP 1: Split data ----
# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], 
    test_size=0.2, 
    random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")


Training samples: 32
Testing samples: 8


In [5]:
# ---- STEP 2: TF-IDF ----
# Converts text into numbers the model can understand
# Each email becomes a vector of word importance scores
vectorizer = TfidfVectorizer(
    max_features=500,    # Only use top 500 most important words
    ngram_range=(1, 2),  # Consider single words AND pairs like "verify immediately"
    stop_words='english' # Ignore common words like "the", "is", "at"
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"\nVocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Feature matrix shape: {X_train_tfidf.shape}")
# Each email is now a row of 500 numbers


Vocabulary size: 405
Feature matrix shape: (32, 405)


In [6]:
# ---- STEP 3: Train the model ----
model = LogisticRegression(random_state=42)
model.fit(X_train_tfidf, y_train)

print("\n✅ Model trained successfully")



✅ Model trained successfully


In [7]:
# ---- STEP 4: Evaluate ----
y_pred = model.predict(X_test_tfidf)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred) * 100:.1f}%")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred, target_names=['Safe', 'Phishing']))


Accuracy: 87.5%

Detailed Report:
              precision    recall  f1-score   support

        Safe       0.75      1.00      0.86         3
    Phishing       1.00      0.80      0.89         5

    accuracy                           0.88         8
   macro avg       0.88      0.90      0.87         8
weighted avg       0.91      0.88      0.88         8



In [8]:
# ---- STEP 5: Test with custom emails ----

def predict_email(text):
    vec = vectorizer.transform([text])
    prediction = model.predict(vec)[0]
    probability = model.predict_proba(vec)[0]
    
    label = "🔴 PHISHING" if prediction == 1 else "🟢 SAFE"
    confidence = max(probability) * 100
    
    print(f"Result: {label}")
    print(f"Confidence: {confidence:.1f}%")
    print()

# Test 1 — obvious phishing
predict_email("Urgent: Verify your account immediately or it will be suspended. Click here now")

# Test 2 — safe email
predict_email("Hi, just wanted to follow up on the report we discussed in yesterday's meeting")

# Test 3 — subtle phishing
predict_email("Your package could not be delivered. Confirm your address at delivery-track.net")

# Test 4 — your own test
predict_email("Dear user, your password expires today. Update it immediately at secure-login.com")

Result: 🔴 PHISHING
Confidence: 61.4%

Result: 🟢 SAFE
Confidence: 61.5%

Result: 🔴 PHISHING
Confidence: 51.6%

Result: 🔴 PHISHING
Confidence: 58.4%



In [9]:
import pandas as pd

df_real = pd.read_csv("sms.tsv", sep='\t', header=None, names=['label', 'text'])

print(f"Total messages: {len(df_real)}")
print(f"\nLabel distribution:")
print(df_real['label'].value_counts())
print(f"\nSample spam:")
print(df_real[df_real['label']=='spam'].iloc[0]['text'])
print(f"\nSample ham:")
print(df_real[df_real['label']=='ham'].iloc[0]['text'])

Total messages: 5572

Label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64

Sample spam:
Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's

Sample ham:
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Convert labels to numbers
df_real['label_num'] = df_real['label'].map({'ham': 0, 'spam': 1})

# Split
X_train, X_test, y_train, y_test = train_test_split(
    df_real['text'], df_real['label_num'],
    test_size=0.2,
    random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# TF-IDF
vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    stop_words='english'
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = model.predict(X_test_tfidf)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred, target_names=['Safe', 'Spam/Phishing']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(f"                Predicted Safe  Predicted Spam")
print(f"Actual Safe         {cm[0][0]}            {cm[0][1]}")
print(f"Actual Spam         {cm[1][0]}            {cm[1][1]}")
print(f"\nFalse Positives (safe flagged as spam): {cm[0][1]}")
print(f"False Negatives (spam missed): {cm[1][0]}")

Training samples: 4457
Testing samples: 1115

Accuracy: 98.03%

Detailed Report:
               precision    recall  f1-score   support

         Safe       0.98      1.00      0.99       966
Spam/Phishing       1.00      0.85      0.92       149

     accuracy                           0.98      1115
    macro avg       0.99      0.93      0.95      1115
 weighted avg       0.98      0.98      0.98      1115


Confusion Matrix:
                Predicted Safe  Predicted Spam
Actual Safe         966            0
Actual Spam         22            127

False Positives (safe flagged as spam): 0
False Negatives (spam missed): 22


In [11]:
import pickle
import os

os.makedirs("model", exist_ok=True)

# Save both the model and vectorizer
pickle.dump(model, open("model/phishing_model.pkl", "wb"))
pickle.dump(vectorizer, open("model/vectorizer.pkl", "wb"))

print("✅ Model saved to model/phishing_model.pkl")
print("✅ Vectorizer saved to model/vectorizer.pkl")

# Verify it loads correctly
loaded_model = pickle.load(open("model/phishing_model.pkl", "rb"))
loaded_vectorizer = pickle.load(open("model/vectorizer.pkl", "rb"))

# Quick test
test = loaded_vectorizer.transform(["Urgent: verify your account immediately or it will be suspended"])
result = loaded_model.predict(test)[0]
print(f"\nVerification test: {'🔴 PHISHING' if result == 1 else '🟢 SAFE'}")

✅ Model saved to model/phishing_model.pkl
✅ Vectorizer saved to model/vectorizer.pkl

Verification test: 🟢 SAFE


In [13]:
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

# ---- Real email phishing patterns ----
email_phishing = [
    "verify your credentials immediately or your account will be permanently deleted",
    "your paypal account has been suspended click here to restore access now",
    "urgent your account will be locked verify your identity at secure-login.com",
    "dear customer your bank account has been compromised enter details immediately",
    "your amazon account suspended confirm credentials at amazon-secure-login.net",
    "security alert unauthorized login detected verify your identity account locked",
    "final notice update payment information immediately or service cancelled",
    "your apple id locked verify information at apple-id-secure.com now",
    "action required unusual signin activity confirm password within 24 hours",
    "your netflix subscription cancelled update billing information immediately",
    "dear user account expires today click here to verify and restore access",
    "you have pending wire transfer confirm bank details to receive funds immediately",
    "your email account deleted unless you verify credentials in 24 hours",
    "microsoft security alert computer infected call support immediately",
    "irs tax refund pending submit social security number to claim refund now",
    "your dhl package on hold pay customs fee at delivery-track.net to release",
    "congratulations you won iphone click here verify shipping address immediately",
    "dear valued customer account suspended update billing information now",
    "urgent password expires today update immediately at secure-login.com",
    "your account access restricted verify identity at account-verify.net now",
    "click here to confirm your paypal payment details before account suspension",
    "your credit card has been charged verify transaction at secure-bank.net",
    "we detected suspicious login from unknown device confirm its you now",
    "your google account will be disabled verify your phone number immediately",
    "important notice your subscription payment failed update card details now",
    "your identity needs verification to unlock your account click the link below",
    "final warning confirm your email address or lose access to your account today",
    "you have unclaimed funds in your account verify details to receive payment",
    "your account has been temporarily suspended please verify your information",
    "urgent action required your account security has been compromised verify now",
    "dear account holder confirm your details to prevent account termination",
    "your package delivery failed confirm your address at fedex-delivery.net",
    "win a free gift card enter your email and password to claim your reward",
    "your computer has a virus download our security tool to remove it now",
    "you have been selected for exclusive rewards verify your details to claim",
    "your login was blocked from a new location verify your account immediately",
    "update your billing information now to avoid interruption of service",
    "we need to verify your account details please click the link to continue",
    "your account will be permanently closed in 24 hours if not verified",
    "confirm your identity to receive your pending bank transfer of funds",
] * 12  # 480 samples

email_safe = [
    "please find attached the quarterly report for your review let me know if questions",
    "can we move the meeting to 3pm tomorrow let me know if that works for you",
    "thanks for joining us at the conference great meeting you lets stay in touch",
    "reminder team standup at 10am today agenda attached for reference",
    "your order has shipped and will arrive by thursday track it on our website",
    "hi wanted to follow up on the project proposal we discussed last week",
    "new version of software now available release notes are in the documentation",
    "please review attached contract and send back comments by end of week",
    "your appointment confirmed for monday at 2pm looking forward to seeing you",
    "great work on the presentation today client was really impressed with analysis",
    "attached is the invoice for last month services payment due within 30 days",
    "can you share the latest version of the spreadsheet when you get a chance",
    "team lunch scheduled for friday at noon restaurant details in calendar invite",
    "just checking in to see how the new project is going need any help from me",
    "your subscription renewed successfully receipt is attached for your records",
    "thanks for your feedback we will look into the issue you mentioned",
    "conference call rescheduled to 4pm dialin details remain the same as before",
    "please complete the compliance training by end of month link is in hr portal",
    "good news budget approved for q2 we can proceed with the project as planned",
    "happy birthday hope you have a wonderful day and a great year ahead",
    "following up on our conversation from last week about the marketing campaign",
    "the design team has finished the mockups let me know what you think",
    "we are pleased to inform you that your application has been accepted",
    "please join us for the annual company picnic this saturday at 2pm",
    "your flight booking confirmation is attached please review the details",
    "the sprint planning meeting has been moved to wednesday at 11am",
    "i wanted to share this article i thought you might find it interesting",
    "the client has approved the proposal we can start next monday",
    "your library book is due back on friday please return or renew it",
    "congratulations on your promotion well deserved looking forward to working together",
    "the server maintenance is scheduled for sunday night from 2am to 4am",
    "please submit your timesheet by end of day friday for payroll processing",
    "the new office wifi password has been updated see attached for details",
    "your dental appointment is tomorrow at 10am please arrive 10 minutes early",
    "the quarterly review is scheduled for next thursday at 3pm in conference room b",
    "we are excited to announce the launch of our new product next month",
    "please review the updated privacy policy and let us know if you have questions",
    "the parking permits for the new building are now available request yours today",
    "your annual performance review has been scheduled for next week",
    "the team has decided to move forward with option b as discussed in the meeting",
] * 12  # 480 samples

# Combine with original SMS dataset
texts_combined = df_real['text'].tolist() + email_phishing + email_safe
labels_combined = df_real['label_num'].tolist() + [1]*len(email_phishing) + [0]*len(email_safe)

df_combined = pd.DataFrame({"text": texts_combined, "label": labels_combined})
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total samples: {len(df_combined)}")
print(df_combined['label'].value_counts())

Total samples: 6532
label
0    5305
1    1227
Name: count, dtype: int64


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    df_combined['text'], df_combined['label'],
    test_size=0.2, random_state=42
)

vectorizer2 = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words='english'
)

X_train_tfidf = vectorizer2.fit_transform(X_train)
X_test_tfidf = vectorizer2.transform(X_test)

model2 = LogisticRegression(random_state=42, max_iter=1000)
model2.fit(X_train_tfidf, y_train)

y_pred = model2.predict(X_test_tfidf)
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(classification_report(y_test, y_pred, target_names=['Safe', 'Phishing']))

# Test your exact phishing email
tests = [
    ("paypal-security@paypal-update.net", "Urgent: Your account has been suspended", "Verify your credentials immediately or your account will be permanently deleted. Click here now."),
    ("hr@company.com", "Team meeting tomorrow", "Hi, just a reminder that we have our weekly standup tomorrow at 10am. See you then."),
    ("noreply@amazon-secure.net", "Action Required", "Your Amazon account has been compromised. Confirm your password at amazon-secure.net immediately."),
]

print("\n--- Live Tests ---")
for sender, subject, body in tests:
    text = f"{sender} {subject} {body}"
    vec = vectorizer2.transform([text])
    pred = model2.predict(vec)[0]
    prob = max(model2.predict_proba(vec)[0]) * 100
    print(f"{'🔴 PHISHING' if pred==1 else '🟢 SAFE'} ({prob:.1f}%) — {subject}")

Accuracy: 98.01%
              precision    recall  f1-score   support

        Safe       0.98      1.00      0.99      1064
    Phishing       1.00      0.90      0.94       243

    accuracy                           0.98      1307
   macro avg       0.99      0.95      0.97      1307
weighted avg       0.98      0.98      0.98      1307


--- Live Tests ---
🔴 PHISHING (98.4%) — Urgent: Your account has been suspended
🟢 SAFE (89.7%) — Team meeting tomorrow
🔴 PHISHING (85.4%) — Action Required
